In [0]:
from pyspark.sql import functions as F

########Generate OrderLine records

In [0]:
from pyspark.sql import functions as F

SEED = 20261022
ORDER_LINE_COUNT = 8000

order_lines = spark.range(1, ORDER_LINE_COUNT + 1)

# Generate unique order line IDs
order_lines = order_lines.withColumn(
    "order_line_id",
    F.concat(
        F.lit("OL-"),
        F.lpad(F.col("id").cast("string"), 8, "0")
    )
)

# Assign valid order IDs
order_lines = order_lines.withColumn(
    "order_number",
    F.pmod(
        F.xxhash64("order_line_id", F.lit(SEED)),
        F.lit(4000)
    ) + 1
)

order_lines = order_lines.withColumn(
    "order_id",
    F.concat(
        F.lit("ORD-2026-"),
        F.lpad(F.col("order_number").cast("string"), 6, "0")
    )
)

# Assign valid product IDs
order_lines = order_lines.withColumn(
    "product_number",
    F.pmod(
        F.xxhash64("order_line_id", F.lit("product")),
        F.lit(20)
    ) + 1
)

order_lines = order_lines.withColumn(
    "product_id",
    F.concat(
        F.lit("PRD-"),
        F.lpad(F.col("product_number").cast("string"), 6, "0")
    )
)

# Generate quantity
order_lines = order_lines.withColumn(
    "quantity",
    (F.pmod(F.xxhash64("order_line_id", F.lit("quantity")), F.lit(5)) + 1).cast("int")
)

# Get actual product prices from Product Master
order_lines = order_lines.join(
    spark.read.table("product_master").select("product_id", "unit_price"),
    on="product_id",
    how="inner"
)

# Generate discount percentage
order_lines = order_lines.withColumn(
    "discount",
    F.when(F.pmod(F.xxhash64("order_line_id", F.lit("discount")), F.lit(100)) < 50, 0)
     .when(F.pmod(F.xxhash64("order_line_id", F.lit("discount")), F.lit(100)) < 80, 5)
     .otherwise(10)
)

# Calculate amounts
order_lines = order_lines.withColumn(
    "gross_amount",
    F.round(F.col("quantity") * F.col("unit_price"), 2)
)

order_lines = order_lines.withColumn(
    "discount_amount",
    F.round(F.col("gross_amount") * F.col("discount") / 100, 2)
)

order_lines = order_lines.withColumn(
    "line_total",
    F.round(F.col("gross_amount") - F.col("discount_amount"), 2)
)

order_line_df = order_lines.select(
    "order_line_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "gross_amount",
    "discount_amount",
    "line_total"
)

display(order_line_df.limit(10))

######Validate OrderLine records


In [0]:
#Check total records:
print("Total OrderLines:", order_line_df.count())

In [0]:
#Check duplicate IDs:
order_line_df.groupBy("order_line_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
#Check invalid order references:
invalid_orders = order_line_df.join(
    spark.read.table("order_master").select("order_id"),
    on="order_id",
    how="left_anti"
)

print("Invalid Order IDs:", invalid_orders.count())

In [0]:
#Check invalid product references:
invalid_products = order_line_df.join(
    spark.read.table("product_master").select("product_id"),
    on="product_id",
    how="left_anti"
)

print("Invalid Product IDs:", invalid_products.count())

In [0]:
#Check financial calculations
order_line_df.select(
    F.sum("gross_amount").alias("Total Gross Amount"),
    F.sum("discount_amount").alias("Total Discount"),
    F.sum("line_total").alias("Total Net Amount")
).show()

In [0]:
#Save the table
order_line_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("order_line")

In [0]:
%sql
SELECT * FROM order_line LIMIT 10;